# HRP — Does it actually beat Markowitz?

Testing López de Prado (2016) on a multi-asset universe. The core claim is that avoiding Σ⁻¹ produces more stable weights without sacrificing much Sharpe.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, leaves_list
from scipy.spatial.distance import squareform

from engine import (
    fetch_market_data,
    estimate_covariance,
    correlation_distance,
    hierarchical_cluster,
    quasi_diagonalize,
    hrp_allocation,
    mean_variance_optimize,
    risk_parity_optimize,
    equal_weight,
    inverse_volatility,
    Backtester,
    compute_metrics,
    effective_number_of_bets,
)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

In [ ]:
tickers = ['SPY', 'QQQ', 'EFA', 'TLT', 'GLD', 'VNQ', 'HYG']
returns = fetch_market_data(tickers, start='2018-01-01')
print(f"{returns.shape[0]} days x {returns.shape[1]} assets")
print(f"{returns.index[0].date()} → {returns.index[-1].date()}")
returns.describe().round(4)

In [ ]:
# covariance + check how ill-conditioned it is
cov, corr = estimate_covariance(returns, method='ledoit_wolf')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title('Correlation Matrix')

kappa = np.linalg.cond(cov.values)
eigvals = np.linalg.eigvalsh(cov.values)
axes[1].bar(range(len(eigvals)), sorted(eigvals, reverse=True))
axes[1].set_xlabel('Eigenvalue Index')
axes[1].set_ylabel('Eigenvalue')
axes[1].set_title(f'Eigenvalue Spectrum (κ = {kappa:.1f})')

plt.tight_layout()
plt.show()
print(f"κ(Σ) = {kappa:.1f} — Markowitz amplifies errors by ~{kappa:.0f}x")

## Clustering

Distance metric: $d(i,j) = \sqrt{(1 - \rho_{i,j})/2}$, then Ward linkage.

In [ ]:
dist = correlation_distance(corr)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pd.DataFrame(dist, index=corr.index, columns=corr.columns),
            annot=True, fmt='.2f', cmap='viridis', ax=axes[0])
axes[0].set_title('Distance Matrix')

Z = hierarchical_cluster(dist, method='ward')
dendrogram(Z, labels=corr.index.tolist(), ax=axes[1], leaf_rotation=45)
axes[1].set_title('Dendrogram (Ward)')
axes[1].set_ylabel('Distance')

plt.tight_layout()
plt.show()

## Seriation

Reorder Σ by dendrogram leaves → block-diagonal structure emerges.

In [ ]:
cov_reordered, sort_order = quasi_diagonalize(cov, Z)
reordered_labels = [corr.index[i] for i in sort_order]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cov, annot=True, fmt='.4f', cmap='viridis', ax=axes[0])
axes[0].set_title('Original Σ')

cov_df_reordered = pd.DataFrame(cov_reordered,
                                 index=reordered_labels,
                                 columns=reordered_labels)
sns.heatmap(cov_df_reordered, annot=True, fmt='.4f', cmap='viridis', ax=axes[1])
axes[1].set_title('Seriated Σ')

plt.tight_layout()
plt.show()

# sanity check: eigenvalues unchanged (it's just a permutation)
eig_orig = sorted(np.linalg.eigvalsh(cov.values))
eig_reord = sorted(np.linalg.eigvalsh(cov_reordered))
print(f"Eigenvalues preserved: {np.allclose(eig_orig, eig_reord)}")
print(f"Ordering: {reordered_labels}")

## Allocation comparison

In [ ]:
w_hrp = hrp_allocation(cov, corr, linkage_method='ward')
w_ew = equal_weight(cov)
w_iv = inverse_volatility(cov)
w_rp = risk_parity_optimize(cov)
w_mv = mean_variance_optimize(cov)

weights_all = pd.DataFrame({
    'HRP': w_hrp, 'EW': w_ew, 'InvVol': w_iv,
    'Risk Parity': w_rp, 'Min Var': w_mv,
})

fig, ax = plt.subplots(figsize=(12, 6))
weights_all.plot(kind='bar', ax=ax)
ax.set_ylabel('Weight')
ax.set_xlabel('Asset')
ax.axhline(y=1/len(tickers), color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\nEffective bets (N_eff):")
for method in weights_all.columns:
    n_eff = effective_number_of_bets(weights_all[method].values)
    print(f"  {method:12s}: {n_eff:.2f} / {len(tickers)}")

## Backtest

252-day rolling window, monthly rebalance. The real test: does HRP hold up OOS?

In [ ]:
methods = {
    'HRP': lambda cov: hrp_allocation(cov, linkage_method='ward'),
    'EW': equal_weight,
    'InvVol': inverse_volatility,
    'Risk Parity': risk_parity_optimize,
    'Min Var': mean_variance_optimize,
}

results = {}
for name, fn in methods.items():
    bt = Backtester(returns, fn, window=252, rebalance_freq=21,
                    cov_method='ledoit_wolf')
    port_ret, weights_hist = bt.run()
    metrics = compute_metrics(port_ret, weights_hist)
    results[name] = {'returns': port_ret, 'weights': weights_hist, 'metrics': metrics}

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for name, res in results.items():
    cum_ret = (1 + res['returns']).cumprod()
    axes[0].plot(cum_ret.index, cum_ret.values, label=name, linewidth=1.5)

axes[0].set_title('Cumulative Returns')
axes[0].set_ylabel('Growth of $1')
axes[0].legend()

for name, res in results.items():
    cum_ret = (1 + res['returns']).cumprod()
    dd = cum_ret / cum_ret.cummax() - 1
    axes[1].plot(dd.index, dd.values, label=name, linewidth=1.5)

axes[1].set_title('Drawdowns')
axes[1].set_ylabel('Drawdown')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
metrics_table = pd.DataFrame({
    name: res['metrics'].to_dict() for name, res in results.items()
})
metrics_table

In [ ]:
# weight evolution: HRP vs MVO
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

results['HRP']['weights'].plot.area(ax=axes[0], alpha=0.7)
axes[0].set_title('HRP weights over time')
axes[0].set_ylim(0, 1)
axes[0].legend(loc='center left', bbox_to_anchor=(1, 0.5))

results['Min Var']['weights'].plot.area(ax=axes[1], alpha=0.7)
axes[1].set_title('Min Variance weights over time')
axes[1].set_ylim(0, 1)
axes[1].legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.tight_layout()
plt.show()

print("Avg monthly turnover:")
for name, res in results.items():
    print(f"  {name:12s}: {res['metrics'].avg_turnover:.4f}")

## Monte Carlo: sensitivity to estimation noise

Perturb Σ with random noise and measure how much each method's weights shift. This is the core argument for HRP — Markowitz is an error maximizer.

In [ ]:
n_simulations = 200
noise_levels = [0.01, 0.05, 0.10, 0.20]

stability_results = {}

for noise in noise_levels:
    weight_diffs = {'HRP': [], 'Min Var': [], 'Risk Parity': []}

    for _ in range(n_simulations):
        perturbation = np.random.randn(*cov.shape) * noise
        perturbation = (perturbation + perturbation.T) / 2
        np.fill_diagonal(perturbation, 0)

        cov_perturbed = cov + perturbation * cov.values
        # force PSD
        eigvals_p, eigvecs_p = np.linalg.eigh(cov_perturbed.values)
        eigvals_p = np.maximum(eigvals_p, 1e-8)
        cov_perturbed = pd.DataFrame(
            eigvecs_p @ np.diag(eigvals_p) @ eigvecs_p.T,
            index=cov.index, columns=cov.columns
        )

        w_hrp_p = hrp_allocation(cov_perturbed)
        w_mv_p = mean_variance_optimize(cov_perturbed)
        w_rp_p = risk_parity_optimize(cov_perturbed)

        weight_diffs['HRP'].append(np.linalg.norm(w_hrp_p.values - w_hrp.values))
        weight_diffs['Min Var'].append(np.linalg.norm(w_mv_p.values - w_mv.values))
        weight_diffs['Risk Parity'].append(np.linalg.norm(w_rp_p.values - w_rp.values))

    stability_results[noise] = {
        method: np.mean(diffs) for method, diffs in weight_diffs.items()
    }

stability_df = pd.DataFrame(stability_results).T
stability_df.index.name = 'Noise Level'

fig, ax = plt.subplots(figsize=(10, 6))
stability_df.plot(marker='o', ax=ax, linewidth=2)
ax.set_xlabel('Noise Level (fraction of Σ magnitude)')
ax.set_ylabel('Mean L2 Weight Deviation')
ax.set_title('Weight Sensitivity to Estimation Error')
ax.legend(title='Method')
plt.tight_layout()
plt.show()

print("At 10% noise:")
for method, dev in stability_results[0.10].items():
    print(f"  {method:12s}: {dev:.4f}")

## Takeaways

- HRP weights are significantly more stable than MVO under estimation noise (which is the point)
- In a bull equity market, EW will beat HRP on raw return (HRP deliberately limits equity cluster exposure)
- HRP's edge is in drawdown control and turnover — check the 2022 drawdown numbers above
- The Monte Carlo confirms the Lipschitz property: small Σ perturbations → small weight changes for HRP

Bottom line: HRP is not a return-maximizing strategy. It's a *risk allocation* that doesn't blow up when your covariance estimate is wrong (which it always is).

---

**Refs**: López de Prado (2016) JPM 42(4); Ledoit & Wolf (2004) JMVA 88(2); Michaud (1989) FAJ 45(1)